In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from dotenv import load_dotenv
load_dotenv('/content/drive/MyDrive/aianalyze.env')
import os
SERVICE_KEY = os.getenv('SERVICE_KEY')
DCLSF_CD = os.getenv('DCLSF_CD', 'A00')
START_DATE = os.getenv('START_DATE', '2020-01-01')
END_DATE = os.getenv('END_DATE', '2025-07-08')
PAGE_SIZE = int(os.getenv('PAGE_SIZE', '1000'))
DRIVE_DIR = os.getenv('DRIVE_DIR', '/content/drive/MyDrive/boan_data')
HF_HOME_DIR = os.getenv('HF_HOME_DIR', '/content/drive/.hf_cache')


In [ ]:
!pip install -q -r requirements.txt
import os
os.environ['HF_HOME'] = HF_HOME_DIR


In [ ]:
import requests

def get_pdf_items():
    items = []
    page = 1
    while True:
        params = {
            'serviceKey': SERVICE_KEY,
            'pageNo': page,
            'numOfRows': PAGE_SIZE,
            'dclsfCd': DCLSF_CD,
            'startDate': START_DATE,
            'endDate': END_DATE,
            'viewType': 'json'
        }
        resp = requests.get('https://api.odcloud.kr/api/genFldPriorInfoDsc/v1/getGenFldList', params=params)
        data = resp.json()
        items.extend(data.get('data', []))
        if data.get('currentCount', 0) < PAGE_SIZE:
            break
        page += 1
    return items

items = get_pdf_items()
print('Fetched', len(items), 'items')


In [ ]:
import tempfile, pdfplumber
import layoutparser as lp

model = lp.models.Detectron2LayoutModel('lp://PubLayNet/faster_rcnn_R_50_FPN_3x/config')

def pdf_to_paragraphs(url):
    out = []
    with tempfile.NamedTemporaryFile(suffix='.pdf') as f:
        r = requests.get(url)
        f.write(r.content)
        f.flush()
        pdf = pdfplumber.open(f.name)
        for page_num, page in enumerate(pdf.pages, start=1):
            words = page.extract_words()
            para = ''
            for w in words:
                para += w['text'] + ' '
                if len(para) >= 150:
                    bbox = [w['x0'], w['top'], w['x1'], w['bottom']]
                    out.append({'page': page_num, 'text': para.strip(), 'bbox': bbox})
                    para = ''
            if para:
                bbox = [words[-1]['x0'], words[-1]['top'], words[-1]['x1'], words[-1]['bottom']]
                out.append({'page': page_num, 'text': para.strip(), 'bbox': bbox})
    return out

paragraphs = []
for item in items:
    url = item.get('download_url')
    if not url:
        continue
    paragraphs.extend([{'pdf_url': url, **p} for p in pdf_to_paragraphs(url)])
print('Extracted', len(paragraphs), 'paragraphs')


In [ ]:
from sentence_transformers import SentenceTransformer
import faiss

try:
    model = SentenceTransformer('upskyy/e5-large-korean')
except Exception:
    model = SentenceTransformer('jinhybr/OOS_KoSimCSE_bert-base')

texts = [p['text'] for p in paragraphs]
vecs = model.encode(texts, batch_size=32, convert_to_numpy=True)
faiss.normalize_L2(vecs)


In [ ]:
import sqlite3, os, shutil
import faiss

conn = sqlite3.connect('docs.db')
cur = conn.cursor()
cur.execute('''CREATE TABLE IF NOT EXISTS docs (id INTEGER PRIMARY KEY, pdf_url TEXT, page INTEGER, text TEXT, b0 REAL, b1 REAL, b2 REAL, b3 REAL)''')
for p, vec in zip(paragraphs, vecs):
    cur.execute('INSERT INTO docs (pdf_url, page, text, b0, b1, b2, b3) VALUES (?,?,?,?,?,?,?)', (p['pdf_url'], p['page'], p['text'], *p['bbox']))
conn.commit()

index = faiss.IndexFlatIP(vecs.shape[1])
index.add(vecs)
faiss.write_index(index, 'faiss_index.faiss')

os.makedirs(DRIVE_DIR, exist_ok=True)
shutil.copy('docs.db', os.path.join(DRIVE_DIR, 'docs.db'))
shutil.copy('faiss_index.faiss', os.path.join(DRIVE_DIR, 'faiss_index.faiss'))


In [ ]:
print('Total PDFs:', len(items))
print('Total paragraphs:', len(paragraphs))
print('SQLite DB:', os.path.join(DRIVE_DIR, 'docs.db'))
print('Faiss index:', os.path.join(DRIVE_DIR, 'faiss_index.faiss'))
